# Greedy vs beam search, per run

Reads the output of `scripts/compare_decoding.py` (submitted by `jobs/compare-decoding.sbatch`):
the same full-data checkpoints decoded on ALMA dev with `num_beams=1` (greedy) and `num_beams=4`.

```
runs_decode_ablation/<job id>/<condition>_nfull_seed<seed>/metrics.json   corpus BLEU / chrF + timing, one entry per beam size
runs_decode_ablation/<job id>/<condition>_nfull_seed<seed>/beam1.jsonl    greedy translations (id, zh, ref, hyp)
runs_decode_ablation/<job id>/<condition>_nfull_seed<seed>/beam4.jsonl    beam-4 translations
```

Sections 1–3 need only `metrics.json`. Section 4 (sentence-level) runs only for runs whose
`beam1.jsonl` and `beam4.jsonl` have been copied back from the cluster.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sacrebleu.metrics import BLEU, CHRF

# Paths below are relative to the repository root.
if Path.cwd().name == "notebooks":
    os.chdir("..")

ABLATION_DIR = Path("runs_decode_ablation")
# Newest job folder by default; set to e.g. ABLATION_DIR / "1083476" to pick one.
JOB_DIR = max((p for p in ABLATION_DIR.iterdir() if p.is_dir()), key=lambda p: p.name)
CONDITIONS = ["uniform", "length", "joint"]
COLOUR = {"uniform": "#2a78d6", "length": "#e07b39", "joint": "#3a9e6f"}
BEAM_NAME = {1: "greedy", 4: "beam4"}
print("job folder:", JOB_DIR)

## 1. Load every run

In [ ]:
rows = []
for metrics_file in sorted(JOB_DIR.glob("*/metrics.json")):
    run_name = metrics_file.parent.name               # e.g. uniform_nfull_seed1
    condition, size, seed = run_name.split("_")
    for m in json.loads(metrics_file.read_text()):
        rows.append({
            "run": run_name,
            "condition": condition,
            "seed": int(seed.removeprefix("seed")),
            "decoding": BEAM_NAME.get(m["num_beams"], f"beam{m['num_beams']}"),
            "bleu": m["bleu"],
            "chrf": m["chrf"],
            "seconds": m["generation_seconds"],
            "n_dev": m["n_dev"],
            "gpu": m["gpu"],
        })
long = pd.DataFrame(rows)

# Signatures should be identical across runs, otherwise the scores are not comparable.
sigs = {(m["bleu_signature"], m["chrf_signature"], m["gpu"])
        for f in JOB_DIR.glob("*/metrics.json") for m in json.loads(f.read_text())}
print(f"found {long['run'].nunique()} runs, {len(long)} (run, decoding) results")
print("distinct signature/GPU combinations:", len(sigs))

In [ ]:
# One row per run, greedy and beam side by side, plus the gain from beam search.
wide = long.pivot_table(index=["condition", "seed"], columns="decoding",
                        values=["bleu", "chrf", "seconds"])
wide.columns = [f"{metric}_{dec}" for metric, dec in wide.columns]
for metric in ["bleu", "chrf"]:
    wide[f"{metric}_gain"] = wide[f"{metric}_beam4"] - wide[f"{metric}_greedy"]
wide["slowdown"] = wide["seconds_beam4"] / wide["seconds_greedy"]
wide["gpu"] = long.groupby(["condition", "seed"])["gpu"].first()
wide = wide.reindex(CONDITIONS, level="condition")
wide.round(3)

## 2. Summary over seeds (mean ± std)

In [ ]:
cols = ["bleu_greedy", "bleu_beam4", "bleu_gain", "chrf_greedy", "chrf_beam4", "chrf_gain", "slowdown"]
summary = wide[cols].groupby(level="condition").agg(["mean", "std"]).reindex(CONDITIONS)
display(summary.round(3))

n_better = {m: int((wide[f"{m}_gain"] > 0).sum()) for m in ["bleu", "chrf"]}
print(f"beam 4 better than greedy: BLEU in {n_better['bleu']}/{len(wide)} runs, "
      f"chrF in {n_better['chrf']}/{len(wide)} runs")

In [ ]:
# Ranking of the conditions under each decoding: does beam search change which condition wins?
for dec in ["greedy", "beam4"]:
    order = summary[(f"bleu_{dec}", "mean")].sort_values(ascending=False)
    print(f"{dec:>6} BLEU ranking:", " > ".join(f"{c} ({v:.2f})" for c, v in order.items()))

## 3. Plots

Left two panels: each line is one run (greedy → beam 4). Right: the gain per run, with the mean per
condition as a bar.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
x = [0, 1]
for ax, metric, label in [(axes[0], "bleu", "BLEU"), (axes[1], "chrf", "chrF")]:
    for (condition, seed), r in wide.iterrows():
        ax.plot(x, [r[f"{metric}_greedy"], r[f"{metric}_beam4"]], marker="o",
                color=COLOUR[condition], alpha=0.75,
                label=condition if seed == wide.loc[condition].index.min() else None)
    ax.set_xticks(x, ["greedy", "beam 4"])
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylabel(f"dev {label}")
    ax.set_title(f"{label}: greedy vs beam 4, per run")
    ax.grid(axis="y", alpha=0.3)
axes[0].legend()

ax = axes[2]
width = 0.38
for i, condition in enumerate(CONDITIONS):
    for j, metric in enumerate(["bleu", "chrf"]):
        gains = wide.loc[condition, f"{metric}_gain"]
        pos = i + (j - 0.5) * width
        ax.bar(pos, gains.mean(), width * 0.9, color=COLOUR[condition],
               alpha=1.0 if metric == "bleu" else 0.5)
        ax.scatter(np.full(len(gains), pos), gains, color="black", s=12, zorder=3)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(len(CONDITIONS)), CONDITIONS)
ax.set_ylabel("beam 4 − greedy")
ax.set_title("Gain per run (solid = BLEU, light = chrF)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cost of beam search: generation time per run (indicative only, not a controlled benchmark).
# Runs landed on different GPUs (full H100 vs a MIG slice), so compare greedy vs beam within a run
# (the slowdown ratio), not absolute seconds across runs. Hatched bars = MIG slice.
fig, ax = plt.subplots(figsize=(7, 3.5))
labels = [f"{c}\nseed{s}" for c, s in wide.index]
pos = np.arange(len(wide))
ax.bar(pos - 0.2, wide["seconds_greedy"], 0.4, label="greedy", color="#9a9a9a")
ax.bar(pos + 0.2, wide["seconds_beam4"], 0.4, label="beam 4", color="#2a78d6")
for p_, gpu in zip(pos, wide["gpu"]):
    if "MIG" in gpu:
        ax.bar([p_ - 0.2, p_ + 0.2], [wide["seconds_greedy"].iloc[p_], wide["seconds_beam4"].iloc[p_]],
               0.4, fill=False, hatch="//", edgecolor="white", lw=0)
ax.set_xticks(pos, labels, fontsize=8)
ax.set_ylabel(f"seconds for {int(long['n_dev'].iloc[0])} dev sentences")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Sentence level (needs `beam1.jsonl` / `beam4.jsonl`)

Copy them from the cluster first, e.g.

```
rsync -av <cluster>:machine_translation/runs_decode_ablation/ runs_decode_ablation/
```

For every run with both files, this computes sentence chrF for greedy and beam, then:
- **paired bootstrap** over sentences of the corpus BLEU / chrF gain (95% interval; sentences only,
  so seed-to-seed variance is not included — compare with the spread in section 2),
- how often beam search changes the output at all, and the win / tie / loss counts by sentence chrF,
- the length ratio (hyp words / ref words): beam search often makes outputs shorter,
- the sentences where beam helps or hurts most, to read.

In [ ]:
def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

pairs = {}
for run_dir in sorted(JOB_DIR.iterdir()):
    g, b = run_dir / "beam1.jsonl", run_dir / "beam4.jsonl"
    if g.exists() and b.exists():
        greedy, beam = read_jsonl(g), read_jsonl(b)
        assert [r["id"] for r in greedy] == [r["id"] for r in beam], run_dir
        pairs[run_dir.name] = pd.DataFrame({
            "id": [r["id"] for r in greedy],
            "zh": [r["zh"] for r in greedy],
            "ref": [r["ref"] for r in greedy],
            "greedy": [r["hyp"] for r in greedy],
            "beam4": [r["hyp"] for r in beam],
        })
HAVE_SENTENCES = bool(pairs)
print(f"sentence-level files for {len(pairs)} of {wide.shape[0]} runs")
if not HAVE_SENTENCES:
    print("-> section 4 is skipped; copy beam1.jsonl / beam4.jsonl back from the cluster.")

In [ ]:
chrf_sent = CHRF()
def words(s):
    return max(len(s.split()), 1)

if HAVE_SENTENCES:
    for name, df in pairs.items():
        for dec in ["greedy", "beam4"]:
            df[f"chrf_{dec}"] = [chrf_sent.sentence_score(h, [r]).score for h, r in zip(df[dec], df["ref"])]
            df[f"len_ratio_{dec}"] = [words(h) / words(r) for h, r in zip(df[dec], df["ref"])]
        df["chrf_diff"] = df["chrf_beam4"] - df["chrf_greedy"]
        df["changed"] = df["greedy"] != df["beam4"]

In [ ]:
def paired_bootstrap(df, n_boot=1000, seed=0):
    """95% interval of corpus BLEU and chrF gain (beam4 − greedy), resampling sentences."""
    rng = np.random.default_rng(seed)
    bleu, chrf = BLEU(tokenize="13a"), CHRF()
    n = len(df)
    refs, g, b = df["ref"].tolist(), df["greedy"].tolist(), df["beam4"].tolist()
    out = {"bleu": [], "chrf": []}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        rr, gg, bb = [refs[i] for i in idx], [g[i] for i in idx], [b[i] for i in idx]
        out["bleu"].append(bleu.corpus_score(bb, [rr]).score - bleu.corpus_score(gg, [rr]).score)
        out["chrf"].append(chrf.corpus_score(bb, [rr]).score - chrf.corpus_score(gg, [rr]).score)
    return {f"{m}_ci": (np.percentile(v, 2.5), np.percentile(v, 97.5)) for m, v in out.items()} | \
           {f"{m}_p_le0": float(np.mean(np.array(v) <= 0)) for m, v in out.items()}

if HAVE_SENTENCES:
    N_BOOT = 1000   # ~1–2 s per 100 resamples per run; lower it for a quick look
    sent_rows = []
    for name, df in pairs.items():
        condition, _, seed = name.split("_")
        boot = paired_bootstrap(df, N_BOOT)
        sent_rows.append({
            "condition": condition, "seed": int(seed.removeprefix("seed")),
            "changed_%": 100 * df["changed"].mean(),
            "beam_wins": int((df["chrf_diff"] > 0).sum()),
            "ties": int((df["chrf_diff"] == 0).sum()),
            "beam_losses": int((df["chrf_diff"] < 0).sum()),
            "len_ratio_greedy": df["len_ratio_greedy"].mean(),
            "len_ratio_beam4": df["len_ratio_beam4"].mean(),
            "bleu_gain_ci": "[{:+.2f}, {:+.2f}]".format(*boot["bleu_ci"]),
            "chrf_gain_ci": "[{:+.2f}, {:+.2f}]".format(*boot["chrf_ci"]),
            "p(bleu gain<=0)": boot["bleu_p_le0"],
            "p(chrf gain<=0)": boot["chrf_p_le0"],
        })
    sentence_summary = pd.DataFrame(sent_rows).set_index(["condition", "seed"]).sort_index()
    display(sentence_summary.round(3))

In [ ]:
# Distribution of the per-sentence chrF change, only over sentences where beam changed the output.
if HAVE_SENTENCES:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for name, df in pairs.items():
        condition = name.split("_")[0]
        d = df.loc[df["changed"], "chrf_diff"]
        axes[0].hist(d, bins=np.arange(-40, 41, 2), histtype="step", color=COLOUR[condition], alpha=0.8)
    axes[0].axvline(0, color="black", lw=0.8)
    axes[0].set_xlabel("sentence chrF, beam 4 − greedy (changed outputs only)")
    axes[0].set_ylabel("sentences")
    axes[0].set_title("Per-sentence change, one line per run")

    # Does beam help more on long sources? Gain by source-length quintile, pooled over runs.
    pooled = pd.concat([df.assign(condition=name.split("_")[0]) for name, df in pairs.items()])
    pooled["src_chars"] = pooled["zh"].str.len()
    pooled["len_q"] = pd.qcut(pooled["src_chars"], 5, labels=[f"Q{i}" for i in range(1, 6)])
    by_len = pooled.groupby(["len_q", "condition"], observed=True)["chrf_diff"].mean().unstack()[
        [c for c in CONDITIONS if c in pooled["condition"].unique()]]
    by_len.plot.bar(ax=axes[1], color=[COLOUR[c] for c in by_len.columns], rot=0)
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].set_xlabel("source length quintile (Q1 = shortest)")
    axes[1].set_ylabel("mean sentence chrF gain")
    axes[1].set_title("Beam gain by source length")
    plt.tight_layout()
    plt.show()

### Sentences to read

Pick a run, then look at where beam search helps and hurts most. Each row shows source,
reference and both outputs.

In [ ]:
RUN = "uniform_nfull_seed0"   # any run name with sentence files
K = 5

if HAVE_SENTENCES:
    df = pairs.get(RUN, next(iter(pairs.values())))
    cols = ["id", "zh", "ref", "greedy", "beam4", "chrf_greedy", "chrf_beam4", "chrf_diff"]
    with pd.option_context("display.max_colwidth", None):
        print("beam 4 helps most")
        display(df.nlargest(K, "chrf_diff")[cols].round(1))
        print("beam 4 hurts most")
        display(df.nsmallest(K, "chrf_diff")[cols].round(1))

In [ ]:
# Sentences where beam helps in (almost) every run: a systematic effect, not seed noise.
if HAVE_SENTENCES and len(pairs) > 1:
    diffs = pd.concat({name: df.set_index("id")["chrf_diff"] for name, df in pairs.items()}, axis=1)
    consistent = pd.DataFrame({
        "runs_beam_better": (diffs > 0).sum(axis=1),
        "runs_beam_worse": (diffs < 0).sum(axis=1),
        "mean_chrf_diff": diffs.mean(axis=1),
    })
    first = next(iter(pairs.values())).set_index("id")
    consistent = consistent.join(first[["zh", "ref", "greedy", "beam4"]])
    with pd.option_context("display.max_colwidth", 120):
        print(f"beam better in all {len(pairs)} runs: {(consistent['runs_beam_better'] == len(pairs)).sum()} sentences; "
              f"worse in all: {(consistent['runs_beam_worse'] == len(pairs)).sum()}")
        display(consistent.sort_values("mean_chrf_diff").head(K))
        display(consistent.sort_values("mean_chrf_diff").tail(K))

## Notes

- Development set only (ALMA dev, 1002 sentences); not test-set results.
- Timing is a single run per setting, and runs were on different GPUs (full H100 and MIG slices);
  treat the slowdown as indicative and do not compare absolute seconds across runs.
- The bootstrap intervals cover sentence sampling only; differences between conditions should be
  judged against the seed spread in section 2 as well.